RANDOM PASSWORD GENERATOR

1. IMPORTS

In [8]:
import secrets
import string
import tkinter as tk
from tkinter import ttk, messagebox

2. CHARACTER SETS

In [9]:
CHARACTER_SETS = {
    "Uppercase": string.ascii_uppercase,
    "Lowercase": string.ascii_lowercase,
    "Numbers": string.digits,
    "Symbols": string.punctuation
}

AMBIGUOUS_CHARACTERS = "0O1lI"

3. PASSWORD GENERATION

In [10]:
def generate_password(length, selected_types, exclude_ambiguous=False):
    """Generate a secure password satisfying the selected rules."""

    if length < 8:
        raise ValueError("Password length must be at least 8.")

    if len(selected_types) < 2:
        raise ValueError("Select at least two character types.")

    character_groups = []

    for category in selected_types:
        if category not in CHARACTER_SETS:
            raise ValueError("Invalid character category selected.")

        characters = CHARACTER_SETS[category]

        if exclude_ambiguous:
            characters = "".join(
                char for char in characters
                if char not in AMBIGUOUS_CHARACTERS
            )

        if not characters:
            raise ValueError(f"No available characters for {category}.")

        character_groups.append(characters)

    if length < len(character_groups):
        raise ValueError(
            "Password length is too short for selected categories."
        )

    all_characters = "".join(character_groups)

    # Guarantee at least one character from each selected category.
    password_chars = [
        secrets.choice(group) for group in character_groups
    ]

    # Fill the remaining positions.
    for _ in range(length - len(password_chars)):
        password_chars.append(secrets.choice(all_characters))

    # Securely shuffle the result.
    secrets.SystemRandom().shuffle(password_chars)

    return "".join(password_chars)


4. PASSWORD STRENGTH CHECKER

In [11]:
def password_strength(password):
    """Estimate strength from length and character diversity."""

    if not password:
        return "Not generated"

    categories_used = sum([
        any(c.islower() for c in password),
        any(c.isupper() for c in password),
        any(c.isdigit() for c in password),
        any(c in string.punctuation for c in password)
    ])

    if len(password) >= 16 and categories_used >= 3:
        return "Strong"
    elif len(password) >= 10 and categories_used >= 2:
        return "Medium"
    else:
        return "Weak"

5. MAIN APPLICATION

In [12]:
class PasswordGeneratorApp:

    def __init__(self, root):
        self.root = root
        self.root.title("Random Password Generator")
        self.root.geometry("540x650")
        self.root.minsize(450, 570)

        self.current_password = ""
        self.history = []

        self.length_var = tk.IntVar(value=12)
        self.upper_var = tk.BooleanVar(value=True)
        self.lower_var = tk.BooleanVar(value=True)
        self.number_var = tk.BooleanVar(value=True)
        self.symbol_var = tk.BooleanVar(value=True)
        self.exclude_var = tk.BooleanVar(value=False)

        self.create_widgets()

    # ------------------------------------------------------
    # GUI LAYOUT
    # ------------------------------------------------------

    def create_widgets(self):

        main = ttk.Frame(self.root, padding=20)
        main.pack(fill="both", expand=True)

        ttk.Label(
            main,
            text="Random Password Generator",
            font=("Arial", 17, "bold")
        ).pack(pady=(0, 8))

        ttk.Label(
            main,
            text="Create secure passwords using your preferences."
        ).pack(pady=(0, 15))

        # Password length
        ttk.Label(
            main, text="Password Length (8–64)"
        ).pack(anchor="w")

        self.length_spin = tk.Spinbox(
            main,
            from_=8,
            to=64,
            textvariable=self.length_var,
            width=8
        )
        self.length_spin.pack(anchor="w", pady=(5, 15))

        # Character categories
        ttk.Label(
            main,
            text="Choose Character Types (minimum 2):",
            font=("Arial", 10, "bold")
        ).pack(anchor="w", pady=(0, 5))

        ttk.Checkbutton(
            main, text="Uppercase letters (A-Z)",
            variable=self.upper_var
        ).pack(anchor="w")

        ttk.Checkbutton(
            main, text="Lowercase letters (a-z)",
            variable=self.lower_var
        ).pack(anchor="w")

        ttk.Checkbutton(
            main, text="Numbers (0-9)",
            variable=self.number_var
        ).pack(anchor="w")

        ttk.Checkbutton(
            main, text="Symbols (!@#$...)",
            variable=self.symbol_var
        ).pack(anchor="w")

        ttk.Checkbutton(
            main,
            text="Exclude ambiguous characters (0, O, 1, l, I)",
            variable=self.exclude_var
        ).pack(anchor="w", pady=(5, 15))

        # Generate button
        ttk.Button(
            main,
            text="Generate Password",
            command=self.generate
        ).pack(fill="x", pady=5)

        # Password display
        ttk.Label(
            main, text="Generated Password:"
        ).pack(anchor="w", pady=(12, 3))

        self.password_var = tk.StringVar()

        self.password_entry = ttk.Entry(
            main,
            textvariable=self.password_var,
            font=("Courier", 12),
            state="readonly"
        )
        self.password_entry.pack(fill="x", pady=3)

        # Strength indicator
        self.strength_label = ttk.Label(
            main, text="Strength: Not generated"
        )
        self.strength_label.pack(anchor="w", pady=5)

        # Copy button
        ttk.Button(
            main,
            text="Copy to Clipboard",
            command=self.copy_password
        ).pack(fill="x", pady=5)

        # History button
        ttk.Button(
            main,
            text="Show Last 5 Passwords",
            command=self.show_history
        ).pack(fill="x", pady=5)

        ttk.Label(
            main,
            text="Passwords are kept in session memory only.",
            font=("Arial", 9)
        ).pack(pady=(10, 0))

    # ------------------------------------------------------
    # GENERATE PASSWORD
    # ------------------------------------------------------

    def generate(self):

        selected_types = []

        if self.upper_var.get():
            selected_types.append("Uppercase")

        if self.lower_var.get():
            selected_types.append("Lowercase")

        if self.number_var.get():
            selected_types.append("Numbers")

        if self.symbol_var.get():
            selected_types.append("Symbols")

        try:
            length = int(self.length_var.get())

            password = generate_password(
                length,
                selected_types,
                self.exclude_var.get()
            )

            self.current_password = password
            self.password_var.set(password)

            self.history.append(password)
            self.history = self.history[-5:]

            strength = password_strength(password)
            self.strength_label.config(
                text=f"Strength: {strength}"
            )

        except (ValueError, tk.TclError) as error:
            messagebox.showerror("Input Error", str(error))

    # ------------------------------------------------------
    # COPY TO CLIPBOARD
    # ------------------------------------------------------

    def copy_password(self):

        if not self.current_password:
            messagebox.showwarning(
                "No Password",
                "Generate a password before copying."
            )
            return

        try:
            self.root.clipboard_clear()
            self.root.clipboard_append(self.current_password)
            self.root.update()

            messagebox.showinfo(
                "Copied",
                "Password copied to clipboard."
            )

        except tk.TclError:
            messagebox.showerror(
                "Clipboard Error",
                "Could not access the clipboard."
            )

    # ------------------------------------------------------
    # PASSWORD HISTORY
    # ------------------------------------------------------

    def show_history(self):

        history_window = tk.Toplevel(self.root)
        history_window.title("Password History")
        history_window.geometry("450x300")

        ttk.Label(
            history_window,
            text="Last 5 Generated Passwords",
            font=("Arial", 12, "bold")
        ).pack(pady=10)

        if not self.history:
            ttk.Label(
                history_window,
                text="No passwords generated yet."
            ).pack(pady=10)
            return

        for index, password in enumerate(
            reversed(self.history), start=1
        ):
            ttk.Label(
                history_window,
                text=f"{index}. {password}",
                font=("Courier", 10)
            ).pack(anchor="w", padx=15, pady=3)

6. START AND RUN THE APPLICATION

In [ ]:
def main():
    root = tk.Tk()
    PasswordGeneratorApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()